In [58]:
print("hallow world")

hallow world


In [59]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from datetime import datetime, timedelta, time
import os
import os
import sys
import pandas as pd
from typing import List, Dict, Optional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings
import random
import re

warnings.filterwarnings("ignore")
import plotly.express as px

In [60]:
dataset_path = r'C:\Users\TPWODL\New folder_Content\AutonomousDataAnalystAgent\data\raw_path\x_data.xlsx'

In [61]:
new_df = pd.read_excel(r'C:\Users\TPWODL\New folder_Content\AutonomousDataAnalystAgent\data\raw_path\x_data.xlsx')

In [62]:
def complaints_status_stacked_bar(dataset_path: str):
    """
    Reads complaint data from Excel and returns a DataFrame.
    Shows Closed vs Open complaints per month, grouped by year.
    """
    # Load and preprocess
    df = pd.read_excel(dataset_path)
    df['Appreciation Tweet'] = (
        df['REMARKS']
        .astype(str)
        .str.findall(r'(?i)\bappreciation\s*tweet\b')
        .apply(lambda x: 'Appreciation Tweet' if x else 'NA')
    )
    df = df[df['Appreciation Tweet'] != 'Appreciation Tweet']

    df['DATE'] = pd.to_datetime(df['DATE'])
    df['YEAR'] = df['DATE'].dt.year
    df['MONTH'] = df['DATE'].dt.month
    df['MONTH_NAME'] = df['DATE'].dt.strftime('%B')
    
    # Group by YEAR, MONTH, MONTH_NAME, CLOSED/OPEN
    summary = (
        df.groupby(['YEAR', 'MONTH', 'MONTH_NAME'])
          .size()
          .reset_index(name='COUNT')
    )
    
    # Sort by YEAR and MONTH to ensure proper chronological order
    summary = summary.sort_values(['YEAR', 'MONTH']).reset_index(drop=True)
    
    return summary

In [63]:
m_df = complaints_status_stacked_bar(dataset_path)

In [64]:
m_df

,YEAR,MONTH,MONTH_NAME,COUNT
0,2022,6,June,249
1,2022,7,July,292
2,2022,8,August,252
3,2022,9,September,202
4,2022,10,October,151
5,2022,11,November,156
6,2022,12,December,257
7,2023,1,January,238
8,2023,2,February,287
9,2023,3,March,437


In [72]:
def complaints_status_stacked(dataset_path: str):
    """
    Reads complaint data from Excel and returns a DataFrame in pivot format.
    Shows Closed vs Open complaints per month, grouped by year.
    Returns a format similar to the provided table with months as columns.
    Filters out appreciation tweets from the data.
    """
    df = pd.read_excel(dataset_path)
    
    # Filter out appreciation tweets
    # Create a boolean mask to identify appreciation tweets
    df['Appreciation Tweet'] = (
        df['REMARKS']
        .astype(str)
        .str.findall(r'(?i)\bappreciation\s*tweet\b')
        .apply(lambda x: 'Appreciation Tweet' if x else 'NA')
    )
    
    # Keep only rows where it's NOT an appreciation tweet
    df = df[df['Appreciation Tweet'] == 'NA']    
    # Load and preprocess
    df['DATE'] = pd.to_datetime(df['DATE'])
    df['YEAR'] = df['DATE'].dt.year
    df['MONTH'] = df['DATE'].dt.month
    df['MONTH_NAME'] = df['DATE'].dt.strftime('%B')
    
    # Group by YEAR, MONTH, MONTH_NAME
    summary = (
        df.groupby(['YEAR', 'MONTH', 'MONTH_NAME'])
          .size()
          .reset_index(name='COUNT')
    )
    
    # Sort by YEAR and MONTH
    summary = summary.sort_values(['YEAR', 'MONTH']).reset_index(drop=True)
    
    # Pivot to create the wide format with months as columns
    pivot_df = summary.pivot_table(
        index='YEAR',
        columns='MONTH_NAME',
        values='COUNT',
        fill_value=0
    )
    
    # Define correct month order
    month_order = ['April', 'May', 'June', 'July', 'August', 'September', 
                   'October', 'November', 'December', 'January', 'February', 'March']
    
    # Reorder columns to match fiscal year (April to March)
    available_months = [m for m in month_order if m in pivot_df.columns]
    pivot_df = pivot_df[available_months]
    
    # Add Total column
    pivot_df['Total'] = pivot_df[available_months].sum(axis=1)
    
    # Reset index to make YEAR a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [73]:
tr = complaints_status_stacked(dataset_path)

In [74]:
tr

MONTH_NAME,YEAR,April,May,June,July,August,September,October,November,December,January,February,March,Total
0,2022,0.0,0.0,249.0,292.0,252.0,202.0,151.0,156.0,257.0,0.0,0.0,0.0,1559.0
1,2023,472.0,564.0,887.0,717.0,751.0,684.0,434.0,274.0,254.0,238.0,287.0,437.0,5999.0
2,2024,798.0,1164.0,1666.0,954.0,867.0,1245.0,656.0,540.0,574.0,227.0,317.0,539.0,9547.0
3,2025,1529.0,2161.0,1272.0,1483.0,1149.0,1316.0,804.0,495.0,506.0,386.0,510.0,691.0,12302.0
4,2026,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0,0.0,0.0,22.0
